# G1 forward / inverse / hybrid playground

Inference only from a compatible three-mode checkpoint. Current exact C3 coefficient representation only; neural outputs are approximate, not certified PDE solves. Default Run All loads no checkpoint and performs no numerical reference solve.

In [ ]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [ ]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    print("1. Verifying repository and preparing dependencies before scientific project imports.")
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    import runpy
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    ENVIRONMENT_SETUP = _reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True)
    _reference_setup["install_project"](BOOTSTRAP["project_dir"], source_revision=BOOTSTRAP["git_commit"])
    print("Dependency preparation complete; no loaded binary was replaced. Existing numerical certification is resolved below; no automatic requalification.")
print("2. Scientific imports and resolved environment.")
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local unified root:", Path(SETUP["local_output_dir"]) / "G1_unified")
print("Drive unified root:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "G1_unified")
print(SETUP["message"])

from pdeep.smooth.bootstrap import environment_versions
print("Python executable:", sys.executable)
print("Installed / loaded dependencies:", json.dumps(environment_versions(), sort_keys=True))

assert all((ROOT / p).is_file() for p in ("src/pdeep/unified/workflow.py", "docs/g1_unified_experiment_v1.md", "notebooks/04_g1_unified.ipynb")), "Configured repository/ref lacks the G1 unified workflow; select its reviewed commit."


In [ ]:
from pdeep.learning_diagnostics.certificates import resolve_certification, copy_certificate, SUPPLIED_ID
CERTIFICATE_ID = SUPPLIED_ID
CERTIFICATE_PATH = None  # Optional full issued JSON path; no upload-path assumption.
USE_EXISTING_LOCAL_QUALIFICATION = not IS_COLAB  # Explicit local policy: reuse existing, never recompute.
CERTIFICATE_CACHE = Path(SETUP["local_output_dir"]) / "certifications/G1_smooth_v1"

ISSUED_CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": CERTIFICATE_ID},
    configured_path=CERTIFICATE_PATH, drive_root=SETUP["drive_root"], cache_dir=CERTIFICATE_CACHE)
display(ISSUED_CERTIFICATION.summary())
CERTIFICATION = ISSUED_CERTIFICATION
if not IS_COLAB and USE_EXISTING_LOCAL_QUALIFICATION and not CERTIFICATION.verified:
    CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": None}, cache_dir=CERTIFICATE_CACHE)
    print("Explicit local policy: selecting an existing native-runtime qualification; issued Linux qualification remains incompatible here.")
    display(CERTIFICATION.summary())
CERTIFICATE = CERTIFICATION.record if CERTIFICATION.verified else None
if CERTIFICATE is None:
    print("Training blocked. Restore the certified runtime/settings or place the full issued JSON in the displayed cache/Drive directory. No requalification has been executed.")
if CERTIFICATION.verified and SETUP["drive_root"]:
    try:
        CERTIFICATE_DRIVE_PATH = copy_certificate(CERTIFICATION.location,
            Path(SETUP["drive_root"]) / "certifications/G1_smooth_v1")
        print("Full certificate byte-copy verified:", CERTIFICATE_DRIVE_PATH)
    except Exception as error:
        print("CERTIFICATE DRIVE COPY FAILED; verified local copy retained:", type(error).__name__, str(error))
PERSISTENCE = SETUP["persistence"]

## Information and inference
Provide exact checkpoint manifest and coefficient values/availability. To generate a reproducible example, a local RNG supplies example coefficients; unknown entries are scrubbed before model conditioning. Optional observations are supplied as the existing Tokens contract. No arbitrary function/sample-completeness claim. Mode labels do not change features; complete inputs determine whether the problem is fully specified.

In [ ]:
import numpy as np
from pdeep.unified_modes.inference import load, infer, plot, generate_example, numerical_reference
CHECKPOINT_MANIFEST = None  # Exact verified three-mode manifest; inverse-only checkpoints rejected.
MODEL_CAPACITY = "large"  # Explicitly choose medium for medium checkpoints.
QUERY_CHUNK_SIZE = 2048
PDE = "wave"
GEOMETRY = "annulus"
MODE = "FORWARD"
PHYSICAL_TIME = 0.2
QUANTITY = 0  #0 displacement/temperature;1 wave velocity.
COEFFICIENT_VALUES = np.random.default_rng(42).standard_normal(64 if PDE == "wave" else 32)
AVAILABLE = np.ones_like(COEFFICIENT_VALUES, dtype=bool) if MODE == "FORWARD" else np.zeros_like(COEFFICIENT_VALUES, dtype=bool)
# For hybrid, explicitly set AVAILABLE for the disclosed source/u0/v0 components.
OBSERVATIONS = None  # Optional existing Tokens with positions, physical times, values, sigma, valid.
GENERATE_EXAMPLE_OBSERVATIONS = False  # Explicit original CPU64 reference solve; can be expensive.
DISPLAY_NUMERICAL_REFERENCE = False  # Explicit full reference, never run by default.
PLAYGROUND_RESULT = None
if CHECKPOINT_MANIFEST:
    CERTIFICATION.require()
    MODEL, METADATA = load(CHECKPOINT_MANIFEST, CERTIFICATE, device=SETUP["training_device"], expected_capacity=MODEL_CAPACITY)
    REFERENCE_COEFFICIENTS = COEFFICIENT_VALUES if AVAILABLE.all() else None
    if GENERATE_EXAMPLE_OBSERVATIONS:
        EXAMPLE = generate_example(PDE, GEOMETRY, CERTIFICATE, mode=MODE, known_fraction=0.5, sensors=32)
        COEFFICIENT_VALUES, AVAILABLE, OBSERVATIONS = EXAMPLE["known_values"], EXAMPLE["available"], EXAMPLE["observations"]
        REFERENCE_COEFFICIENTS = EXAMPLE["reference_coefficients"]  # Scoring only, never conditioning.
    PLAYGROUND_RESULT = infer(MODEL, PDE, GEOMETRY, COEFFICIENT_VALUES, AVAILABLE,
        observations=OBSERVATIONS, mode=MODE, query_chunk=QUERY_CHUNK_SIZE)
    display({"checkpoint":METADATA, "status":PLAYGROUND_RESULT["status"],
             "known_fraction":PLAYGROUND_RESULT["known_fraction"], "seconds":PLAYGROUND_RESULT["seconds"]})
    NUMERICAL_REFERENCE = None
    if DISPLAY_NUMERICAL_REFERENCE:
        assert REFERENCE_COEFFICIENTS is not None, "Reference needs the complete true inputs; use a generated example or fully supplied field."
        DESIGN = (PLAYGROUND_RESULT["position"], PLAYGROUND_RESULT["time"], PLAYGROUND_RESULT["quantity"], PDE != "poisson")
        NUMERICAL_REFERENCE = numerical_reference(PDE, GEOMETRY, REFERENCE_COEFFICIENTS, DESIGN, CERTIFICATE)
    display(plot(PLAYGROUND_RESULT, physical_time=PHYSICAL_TIME, quantity=QUANTITY, reference=NUMERICAL_REFERENCE))
else:
    print("Supply a compatible verified checkpoint to run inference. No training or numerical solve executed.")